# Lab 5 — Security in FastAPI (OAuth2 + JWT for Multi-Tenant AI Access)

Difficulty: Advanced | ~45-60 min


### What You'll Build

A bare session system trusts any client who knows a `session_id` — whoever holds that string can use it. This lab locks access down: users log in through a real OAuth2 password flow, receive a signed JWT, and every chat request must present that token. The tenant (alice → `tenant-a`, bob → `tenant-b`) is read out of the **verified** token — never from anything the client claims — and each tenant's history is keyed by a composite `(tenant_id, session_id)` pair.

The payoff is demonstrated with real code: two tenants using the **identical** `session_id` string still get completely isolated conversation histories.


### Step 0: Install Dependencies

One cell installs every pinned dependency. Two are new: `PyJWT` for signing and verifying tokens, and `pwdlib` (with the argon2 backend) for password hashing. `python-multipart` is required because OAuth2 login is sent as form data, not JSON.


In [1]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 python-dotenv==1.2.3 openai==3.5.0 PyJWT==2.12.0 "pwdlib[argon2]==0.3.1" python-multipart==0.0.32



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Imports, API Key, and App Setup

`load_dotenv()` reads the existing `.env`; `os.getenv("OPEN_ROUTER_KEY")` gives the OpenRouter key. The one new environment-driven value is the JWT signing secret. The `JWT_SECRET` fallback string below is a lab convenience so the notebook runs without extra setup — in production the secret must come from an environment variable or a secrets manager, never source code. `conversation_store` is a plain in-memory dict whose keys will now be `(tenant_id, session_id)` tuples.


In [2]:
from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from dotenv import load_dotenv
from openai import AsyncOpenAI
from typing import Annotated
import jwt
import os
import time

load_dotenv()

api_key = os.getenv("OPEN_ROUTER_KEY")
if not api_key:
    api_key = input("Open Router API key: ")

# Production rule: JWT secret from env/secrets manager, NEVER hardcoded.
# The default is a lab-only convenience so the notebook runs as-is.
JWT_SECRET = os.getenv("JWT_SECRET", "lab5-dev-only-secret-not-for-production")

conversation_store = {}   # keys are (tenant_id, session_id) tuples
app = FastAPI()


### Step 2: Password Hashing and the User Store

`PasswordHash.recommended()` bundles **argon2id** — a deliberately slow, memory-hard key-derivation function built exactly for password storage. Hashing happens at startup, so the store holds hash only, never plaintext. alice and bob are hardcoded stand-ins for what would be a database lookup in production; the shape is what matters: username → hashed password → `tenant_id`. Notice that both of alice and bob's rows are distinct — one tenant each, which is the precondition for the tenant isolation.


In [3]:
from pwdlib import PasswordHash

password_hash = PasswordHash.recommended()

users = {
    "alice": {
        "password_hash": password_hash.hash("alice-password"),
        "tenant_id": "tenant-a",
    },
    "bob": {
        "password_hash": password_hash.hash("bob-password"),
        "tenant_id": "tenant-b",
    },
}

print("alice password_hash:", users["alice"]["password_hash"])


alice password_hash: $argon2id$v=19$m=65536,t=3,p=4$XbGUEy+XamMzmpqeJ5YyFw$8R6DacDV1qj6prhFmdPa8UCiQGBn37suutSATr+j7o4


### Step 3: The Token Scheme and Token-Issuing Function

`OAuth2PasswordBearer(tokenUrl="/token")` is FastAPI's declarative way to mark endpoints as protected: it advertises that a bearer token is obtained from a path,`/token` in our case, and as a dependency it parses `Authorization: Bearer <token>` and hands you the raw token string. It does **not** verify the token — verification is Step 5's job.

`create_access_token()` signs a three-claim payload with HS256 over our secret: `sub` (the username), `tenant_id`, and `exp`. The `expires_in_seconds` parameter defaults to 1800 so the same function can mint the short-lived tokens the expiry demo needs later, without duplicating code.


In [4]:
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/token")

def create_access_token(username, tenant_id, expires_in_seconds=1800):
    payload = {
        "sub": username,
        "tenant_id": tenant_id,
        "exp": time.time() + expires_in_seconds,
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


### Step 4: The `/token` Login Endpoint

`OAuth2PasswordRequestForm` is FastAPI's parser for the standard OAuth2 password flow: the client sends `username` and `password` as **form-encoded** body (`application/x-www-form-urlencoded`), not JSON. That is also why `tokenUrl` was set above — OAuth2-speaking clients know to POST form data to that path and read `access_token` from the JSON reply.

The check is one combined condition: the username must exist **and** `password_hash.verify()` must accept the supplied password against the stored hash. The timing guard means an argon2 verify runs even when the username is unknown, so an attacker cannot measure response time to learn which accounts exist. Fail either side → 401. Success → a signed token for the user's real tenant.


In [ ]:
@app.post("/token")
async def login(form_data: Annotated[OAuth2PasswordRequestForm, Depends()]):
    stored = users.get(form_data.username)
    # Timing guard: ALWAYS run one argon2 verify, even for unknown usernames,
    # so response time cannot reveal which accounts exist.
    target = stored["password_hash"] if stored else users["alice"]["password_hash"]
    valid_password = password_hash.verify(form_data.password, target)

    if not stored or not valid_password:
        raise HTTPException(status_code=401, detail="Incorrect username or password")
        
    token = create_access_token(form_data.username, stored["tenant_id"])
    return {"access_token": token, "token_type": "bearer"}


### Step 5: The Token-Verification Dependency

Here the token stops being a string and becomes an identity. `get_current_tenant` depends on `oauth2_scheme` (which extracts the bearer token), then lets `jwt.decode()` do the honesty check. `decode()` verifies the signature **and** the `exp` claim, raising its own exceptions on failure — `ExpiredSignatureError` when `exp` is past, `InvalidSignatureError`/`DecodeError` when the token is forged or garbled. We catch the common parent `jwt.PyJWTError` and collapse every failure into a 401. There is no hand-rolled expiry comparison anywhere in this notebook.

Note too: if the header is missing entirely, `oauth2_scheme` itself raises a 401 before `decode()` is even reached.


In [6]:
async def get_current_tenant(token: Annotated[str, Depends(oauth2_scheme)]):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(status_code=401, detail="Invalid or expired token")
    return payload["tenant_id"]


### Step 6: Tenant-Scoped Session History

If history were keyed by `session_id` alone, anyone who owned that string would own the conversation. This dependency builds a **composite key** `(tenant_id, session_id)`. `tenant_id` comes from `get_current_tenant` — i.e. from inside the verified token — while `session_id` is an ordinary query parameter the client is free to pick. Two tenants submitting the same `session_id` land on different keys, and therefore different, isolated histories.


In [7]:
async def get_session_history(
    tenant_id: Annotated[str, Depends(get_current_tenant)],
    session_id: str,
):
    key = (tenant_id, session_id)
    if key not in conversation_store:
        conversation_store[key] = []
    return conversation_store[key]


### Step 7: Shared Client and the Protected `/chat` Endpoint

The client dependency is one shared `AsyncOpenAI` instance holding the connection pool and auth. The `/chat` endpoint appends the user message, calls the LLM, stores the reply, and returns it — but two things changed: `history` no longer comes from a freely-chosen session key, and every request must carry a valid bearer token or the whole chain dies in `get_session_history` before `chat()` ever runs.


In [8]:
client = AsyncOpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

async def get_client():
    return client

@app.post("/chat")
async def chat(
    message: str,
    session_id: str,
    history: Annotated[list, Depends(get_session_history)],
    client: Annotated[AsyncOpenAI, Depends(get_client)],
):
    history.append({"role": "user", "content": message})

    response = await client.chat.completions.create(
        model="openrouter/free",
        messages=history,
    )

    answer = response.choices[0].message.content
    history.append({"role": "assistant", "content": answer})

    return {"answer": answer, "history": history}


### Step 8: TestClient

A single test client drives every demo. One quirk worth knowing: because `/chat` awaits an external async client, running `TestClient` repeatedly in a single notebook session can occasionally raise an "Event loop is closed" error. There is no clean one-line fix worth adding here — if it appears, simply re-run that cell; a fresh event loop is created on retry.


In [9]:
test_client = TestClient(app)


### Demo 1: Login as alice → a real token

POST `/token` with alice's credentials as form data. We decode the returned token with the same secret to show its claims: `sub=alice`, `tenant_id=tenant-a`, and a numeric `exp` about 30 minutes out.


In [10]:
res = test_client.post(
    "/token",
    data={"username": "alice", "password": "alice-password"},
)

print("status:", res.status_code)
alice_token = res.json()["access_token"]
print("token:", alice_token[:35] + "...")
print("claims:", jwt.decode(alice_token, JWT_SECRET, algorithms=["HS256"]))


status: 200
token: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ...
claims: {'sub': 'alice', 'tenant_id': 'tenant-a', 'exp': 1788266315.184957}


### Demo 2: alice chats on session "shared-abc"

alice sends the first message of a conversation. The history is created under key `("tenant-a", "shared-abc")` and she gets a real LLM answer.


In [11]:
res = test_client.post(
    "/chat",
    params={"message": "Hello. Where is Paris?", "session_id": "shared-abc"},
    headers={"Authorization": f"Bearer {alice_token}"},
)

print("status:", res.status_code)
print("answer:", res.json()["answer"])


status: 200
answer: 

Paris is the capital city of France, located in the north-central part of the country. It sits along the Seine River, about 230 miles (370 km) southwest of London, and is known worldwide for its art, fashion, gastronomy, and culture.


### Demo 3: Follow-up question on the same session

Same token, same `session_id`. The endpoint passes the full prior history into the LLM, so the follow-up — "And what is its population?" — resolves against the Paris context. Proof that context really carries across calls — now inside a single tenant. The history length grows to 4 messages (2 user + 2 assistant).


In [12]:
res = test_client.post(
    "/chat",
    params={"message": "And what is its population?", "session_id": "shared-abc"},
    headers={"Authorization": f"Bearer {alice_token}"},
)

print("answer:", res.json()["answer"])
print("history length:", len(res.json()["history"]))


answer: Paris — the city proper (the “commune”) has a population of roughly **2.1 million people** (about 2,165,000 as of 2023). The larger Paris metropolitan area, which includes the surrounding suburbs, is home to around **12 million** people.
history length: 4


### Demo 4: Login as bob → a different tenant

bob logs in and receives his own token. Decoding it shows `sub=bob`, `tenant_id=tenant-b`. Next, bob is going to reuse alice's exact `session_id` string — which is where the Lab's isolation promise gets tested.


In [13]:
res = test_client.post(
    "/token",
    data={"username": "bob", "password": "bob-password"},
)

print("status:", res.status_code)
bob_token = res.json()["access_token"]
print("claims:", jwt.decode(bob_token, JWT_SECRET, algorithms=["HS256"]))


status: 200
claims: {'sub': 'bob', 'tenant_id': 'tenant-b', 'exp': 1788266324.4623575}


### Demo 5: bob, the SAME session_id → isolated history

bob submits the identical `session_id` string alice used. Yet his history starts empty: his conversation is keyed under `("tenant-b", "shared-abc")`, a different key, so he can neither see nor continue alice's thread. The printed store keys make the composite-key design visible — both tenants have a "shared-abc" session, and neither can reach the other's. After bob's single message his history has 2 entries; alice's still has 4.


In [14]:
res = test_client.post(
    "/chat",
    params={"message": "Hello. Do you know Paris?", "session_id": "shared-abc"},
    headers={"Authorization": f"Bearer {bob_token}"},
)

print("answer:", res.json()["answer"])
print("bob's history length:", len(res.json()["history"]))
print("alice's history length:", len(conversation_store[("tenant-a", "shared-abc")]))
print("store keys:", list(conversation_store.keys()))


answer: Hello! Yes, I know Paris very well. It's the capital of France, world-famous for landmarks like the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, and the Champs-Élysées. It's also celebrated for its art, fashion, cuisine, and romantic atmosphere. 

Is there something specific about Paris you're interested in—history, travel tips, culture, or maybe something else?
bob's history length: 2
alice's history length: 4
store keys: [('tenant-a', 'shared-abc'), ('tenant-b', 'shared-abc')]


### Demo 6: Wrong password → 401

The whole point of hashing is that verification runs against the stored argon2id hash, and a wrong guess fails. `/token` returns 401 and no token. This also shows why `verify()` (constant-time) exists instead of comparing plaintext — an attacker who grabs the database still cannot learn the password or log in.


In [15]:
res = test_client.post(
    "/token",
    data={"username": "alice", "password": "wrong-password"},
)

print("status:", res.status_code)
print("body:", res.json())


status: 401
body: {'detail': 'Incorrect username or password'}


### Demo 7: Garbage bearer token → 401

Feed the protected endpoint a malformed token. `oauth2_scheme` extracts "garbage.token.here" fine, but `jwt.decode()` immediately raises `DecodeError` (a `PyJWTError` subclass), our dependency converts it to 401, and the request never reaches `chat()`. No manual string inspection — pyjwt's own validation did the rejecting.


In [16]:
res = test_client.post(
    "/chat",
    params={"message": "do not answer", "session_id": "shared-abc"},
    headers={"Authorization": "Bearer garbage.token.here"},
)

print("status:", res.status_code)
print("body:", res.json())


status: 401
body: {'detail': 'Invalid or expired token'}


### Demo 8: Expiry — pyjwt enforces `exp`, not us

One cell covers the whole lifecycle: mint a token with `expires_in_seconds=10`, use it successfully, `time.sleep(10)` past its expiry, then use the **exact same token** again. It now fails with 401 — because `jwt.decode()` checks the `exp` claim itself and raises `ExpiredSignatureError` (a `PyJWTError` subclass), which our single `except jwt.PyJWTError` catches. There is no `if time.time() > exp:` anywhere in this notebook; the library owns the check. The final lines decode the crashed token directly to show the exception class pyjwt raised.


In [17]:
short_token = create_access_token("alice", "tenant-a", expires_in_seconds=10)

res_before = test_client.post(
    "/chat",
    params={"message": "I am still valid", "session_id": "expiry-demo"},
    headers={"Authorization": f"Bearer {short_token}"},
)
print("before expiry - status:", res_before.status_code)

time.sleep(10)

res_after = test_client.post(
    "/chat",
    params={"message": "Still valid?", "session_id": "expiry-demo"},
    headers={"Authorization": f"Bearer {short_token}"},
)
print("after expiry  - status:", res_after.status_code)
print("after expiry  - body:", res_after.json())

try:
    jwt.decode(short_token, JWT_SECRET, algorithms=["HS256"])
except jwt.PyJWTError as e:
    print("jwt.decode raised:", type(e).__name__, "-", e)


before expiry - status: 200


after expiry  - status: 401
after expiry  - body: {'detail': 'Invalid or expired token'}
jwt.decode raised: ExpiredSignatureError - Signature has expired
